# 04 · Phenology metrics per year and topographic zone

Loads the cached Sentinel-2 statistics, filters dates by quality and derives the start, peak and end of the autumn yellowing season for every year and zone with two methods, the Whittaker smoother and the double logistic, each fitted to every index. It compares the two, projects the chosen dates a few years ahead and relates them to elevation and aspect.

| Section | Content |
|---|---|
| 1 | load and filter the cached statistics |
| 2 | the indices over the years |
| 3 | Whittaker and double-logistic curves per year and zone |
| 4 / 5 / 6 | start / peak / end of the season: both methods |
| 7 | comparison of the two methods and the chosen date per date type |
| 8 | OLS trend and projection of the chosen dates |
| 9 | chosen dates vs elevation and aspect |
| 10 | STL decomposition of the indices |
| 11 | export |

- **Settings:** `config.py` (years, date filters, fit settings, `SEASON_INDICES` and `SEASON_CHOICE`)
- **Reads:**
  - `cache/<AOI>_zones/` (from `01_download_sentinel2`)
  - `cache/validation/<AOI>/` (from `02_download_validation`, optional)
- **Writes** to `results/<AOI>/`:
  - `season_dates.csv`: start / mid / end of both methods per index, zone and year, and the combined dates per date type, with a `trusted` flag
  - `season_method_comparison.csv`: the comparison of section 7
  - `season_selected.csv`: the chosen start / peak / end per zone and year (read by `05_climate_analysis`)
  - `season_projection.csv`: the projection of section 8
  - `zones.csv`, `topography_effects.csv`
  - `ndvi_timeseries_all_dates.csv`, `ndvi_timeseries_clean.csv`

Years whose autumn is not fully cached yet (`SEASON_COMPLETE_DOY`, e.g. the current year) are shown as preliminary in the plots but left out of all tables, trends and models.

Runs offline in about a minute. Sources are listed at the end.

## Yellowing season: start, peak and end

The aim is three dates per year and zone. Both methods, the Whittaker smoother and the double logistic, are fitted to every index (`INDICES`). Each date type is then read from the indices whose signal matches it (`SEASON_INDICES` in `config.py`). A method's date is the median of its dates over these indices, which gives one **candidate** per method:

| Date | Meaning | Indices | Share of the autumn amplitude left | Checked against (section 7) |
|---|---|---|---|---|
| **Start** | needles begin to lose chlorophyll | GRVI, RCC | 90 % | order, data support |
| **Peak** | canopy most yellow: chlorophyll gone, needles still on | GRVI, RCC | 50 % | MeteoSwiss needle colouring 50 % |
| **End** | needles have fallen | NDVI, GRVI, RCC | 15 % | MeteoSwiss needle drop 50 % |

- **Indices and what they see:**
  - **GRVI** (green − red) / (green + red) falls and **RCC** red / (red + green + blue) rises as chlorophyll breaks down: the change from green to yellow shows mainly in the visible bands (Motohka et al. 2010; RCC is the standard PhenoCam index, Richardson et al. 2018). They mark the **colour change**, so they set the start and the peak. Two indices instead of one make the date less dependent on the noise of a single index; `index_spread` shows how far they disagree.
  - **NDVI** also responds to the red band, but yellow needles still reflect NIR strongly, so it drops mostly with **needle fall**. On its own, however, its Whittaker end lies mostly in the November data gap and agrees poorly with the MeteoSwiss needle drop (r ≈ 0.2, section 7). The colour indices also reach their post-season level once the needles are gone, so the **end** is the median of NDVI, GRVI and RCC: a median of three is not pulled away by one index that fails.
  - **PSRI** (Merzlyak et al. 1999) is fitted and compared in section 7 but not used: on the multidate AOI its dates come later at higher elevation, opposite to the expected gradient.
  - Every single-index date is kept in `season_dates`, so section 7 shows how each index alone would score.
- **Why these two methods:** an earlier version of this notebook compared six methods on both AOIs (git commit `3947717`). The Whittaker dates ranked best for every date type: closest to the MeteoSwiss stations, most often backed by clear observations, and in the order start < peak < end. The double logistic, the function TIMESAT fits for HR-VPP, was the best parametric method and gives an independent second estimate. Dropped:
  - the free autumn logistic: fits often on a bound, weakest MeteoSwiss agreement
  - the fixed-base logistic: bootstrap intervals of 40–90 days
  - the third derivative of the double logistic (Busetto et al. 2010): dates and ranks close to the double logistic itself
  - the NDVI moving-average baselines: no MeteoSwiss agreement for the midpoint (r ≈ 0); the fixed threshold is reached in only half of the zone-years
  - the yellowness maximum as peak: after the end in most zone-years
- **Thresholds:** 90 / 50 / 15 % of the amplitude follow MODIS MCD12Q2 (senescence / greendown midpoint / dormancy; Gray et al. 2019) and HR-VPP (end of season at 15 %; Tian et al. 2021).
- **Expected:** start < peak < end in every year and zone; earlier at higher elevation.
- **Limits:** after mid-October there are few clear observations (low sun turns north slopes SCL "dark", clouds, early snow). Every date therefore carries a `trusted` flag (3d); the figures of sections 4–6 show dates that are not trusted hollow.

In [ ]:
from datetime import date, timedelta

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings

from matplotlib.colors import LightSource
from scipy import sparse
from scipy.optimize import OptimizeWarning, curve_fit
from scipy.sparse.linalg import spsolve
from statsmodels.tsa.seasonal import STL

import importlib
import config, phenology_utils
importlib.reload(config); importlib.reload(phenology_utils)  # pick up edits to config.py without a kernel restart
from config import *
from phenology_utils import clean_frame, incomplete_years, index_frame, load_stats, load_zone_table

INDEX_COLORS = {"NDVI": "darkgreen", "GRVI": "darkorange", "PSRI": "purple", "RCC": "firebrick"}
INDEX_MARKERS = {"NDVI": "o", "GRVI": "s", "PSRI": "^", "RCC": "v"}
ASPECT_CODES = {"N": 0, "S": 1, "flat": 2}   # aspect classes as stored in zones.tif (01)
ASPECT_COLORS = {"N": "tab:blue", "S": "tab:orange", "flat": "grey"}
METHOD_COLORS = {"Whittaker": "tab:green", "double logistic": "tab:orange"}
print(f"AOI: {AOI_NAME} | years {FIRST_YEAR}-{LAST_YEAR} | results: {OUT_DIR}")

## 1. Load and filter the cached statistics

A date is kept for a zone if at least `MIN_VALID_FRACTION` of the zone's pixels are clear (and it passes `MAX_SCENE_CLOUD` / `MAX_SCL_FRACTION`). With `VALID_SHARE_OF = "covered"`, this and the other SCL shares count only the zone pixels inside the scene's footprint: every second scene comes from the edge of a neighbouring orbit's swath and covers only part of the AOI, and such a scene now passes if its covered part is clear and holds at least `MIN_COVERED_PIXELS` pixels. `df`, `df_clean` and so on refer to the whole AOI (zone `"all"`). `clean_by_zone[index][zone]` holds the cleaned median series of every zone. `INCOMPLETE` lists the years whose cache ends before `SEASON_COMPLETE_DOY`. Indices of `INDICES` that the cache lacks (e.g. RCC before `01` was re-run) are skipped.

In [ ]:
stats_all = load_stats()
zone_table = load_zone_table()
ZONES = list(zone_table.index)   # "all" first, then elevation band x aspect
INCOMPLETE = incomplete_years(stats_all)
if INCOMPLETE:
    print(f"Autumn not fully cached (preliminary, left out of tables and models): {INCOMPLETE}")

missing = [n for n in INDICES if f"{n.lower()}_median" not in stats_all or stats_all[f"{n.lower()}_median"].isna().all()]
if missing:
    print(f"Not in the cache, skipped: {missing} (re-run 01_download_sentinel2 to add them)")
INDICES = [n for n in INDICES if n not in missing]

frames = {name: index_frame(stats_all, name.lower()) for name in INDICES}
clean = {name: clean_frame(stats_all, name.lower()) for name in INDICES}
df, df_clean = frames["NDVI"], clean["NDVI"]
for name in INDICES:
    print(f"{name}: {len(clean[name])} of {len(frames[name])} dates kept")

clean_by_zone = {name: {zone: clean_frame(stats_all, name.lower(), zone)[f"{name.lower()}_median"] for zone in ZONES}
                 for name in INDICES}
display(zone_table[["n_pixels", "elev_mean", "slope_mean", "northness_mean"]].round(2).join(
    pd.DataFrame({name: {z: len(s) for z, s in by_zone.items()} for name, by_zone in clean_by_zone.items()})
      .add_prefix("dates_")))

### 1b. Check: partial-swath scenes vs full scenes

A partial scene always sees the same strip of a zone, from a different viewing angle. If that strip is not representative, its zone median is offset from the full-scene medians. Each kept partial scene (covering < 90 % of the zone) is paired with the nearest kept full scene at most `PAIR_MAX_DAYS` days away (usually 2), both without snow.

- **`bias`:** mean difference partial − full; near 0 = no systematic offset
- **`mad`:** median absolute difference, including real change over the days in between
- **`full_mad`:** the same for pairs of full scenes ≤ 5 days apart, as the noise to compare against
- **`bias_share`:** `bias` relative to the zone's seasonal range (5th–95th percentile)

A `bias_share` of a few percent shifts a transition date by much less than a day. If it is large for a zone, set `VALID_SHARE_OF = "zone"` or raise `MIN_COVERED_PIXELS`.

In [ ]:
PAIR_MAX_DAYS = 3
PARTIAL_COVER = 0.9   # scenes covering less of the zone count as partial


def paired_diff(a, b, max_days):
    # a - b for each date of a and the nearest other date of b at most max_days away (series indexed by date)
    m = pd.merge_asof(a.rename("a").to_frame(), b.rename("b").to_frame(), left_index=True, right_index=True,
                      direction="nearest", tolerance=pd.Timedelta(days=max_days), allow_exact_matches=False)
    return (m.a - m.b).dropna()


rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)   # pandas' merge_asof tolerance on recent numpy
    for name in INDICES:
        col = f"{name.lower()}_median"
        for zone in ZONES:
            f = clean_frame(stats_all, name.lower(), zone)
            f = f[f.scl_snow_fraction <= MAX_SNOW_FRACTION]
            partial = f.covered_fraction < PARTIAL_COVER
            diff = paired_diff(f.loc[partial, col], f.loc[~partial, col], PAIR_MAX_DAYS)
            full_diff = paired_diff(f.loc[~partial, col], f.loc[~partial, col], 5)
            spread = f[col].quantile(0.95) - f[col].quantile(0.05)
            rows.append({"index": name, "zone": zone, "partial_kept": int(partial.sum()), "n_pairs": len(diff),
                         "bias": diff.mean(), "mad": diff.abs().median(), "full_mad": full_diff.abs().median(),
                         "bias_share": diff.mean() / spread})
partial_check = pd.DataFrame(rows).set_index(["index", "zone"])
print(f"VALID_SHARE_OF = {VALID_SHARE_OF!r}, MIN_COVERED_PIXELS = {MIN_COVERED_PIXELS}")
display(partial_check.round(3))

## 2. The indices over the years (whole AOI)

- **Top:** every index over the whole period: all dates (grey), the dates that pass the filter (colour) and a centred moving average (`SMOOTH_WINDOW` × 5 days) on a regular 5-day grid, with gaps of up to 30 days interpolated. Shaded: preliminary years.
- **Bottom:** the moving average of each year over the snow-free season (`SEASON_WINDOW`), one line per year.

The moving average is only for display; the date methods work on the observations directly.

In [ ]:
def smooth(series):
    # Centred moving average on a regular 5-day grid; gaps of up to 30 days interpolated, longer ones left empty
    regular = series.resample("5D").mean().interpolate(method="time", limit=6)
    rolled = regular.rolling(SMOOTH_WINDOW, center=True, min_periods=3).mean()
    return rolled.where(regular.notna())


def clear_limits(name, margin=0.15):
    # y limits from the clear dates (snow and cloud values of the unfiltered dates would squash the plot)
    lo, hi = medians[name].quantile([0.005, 0.995])
    return lo - margin * (hi - lo), hi + margin * (hi - lo)


medians = {name: clean[name][f"{name.lower()}_median"] for name in INDICES}
smoothed = {name: smooth(s) for name, s in medians.items()}

fig, axes = plt.subplots(len(INDICES), 1, figsize=(13, 2.4 * len(INDICES)), sharex=True, squeeze=False)
for ax, name in zip(axes[:, 0], INDICES):
    ax.scatter(frames[name].index, frames[name][f"{name.lower()}_median"], s=5, color="lightgrey", label="all dates")
    ax.scatter(medians[name].index, medians[name], s=8, color=INDEX_COLORS[name], label="clear dates")
    ax.plot(smoothed[name].index, smoothed[name], color="black", lw=1.2,
            label=f"{SMOOTH_WINDOW * 5}-day moving average")
    for year in INCOMPLETE:
        ax.axvspan(pd.Timestamp(year, 1, 1), pd.Timestamp(year, 12, 31), color="grey", alpha=0.1, lw=0)
    ax.set_ylim(*clear_limits(name))
    ax.set_ylabel(name)
    ax.grid(alpha=0.3)
axes[0, 0].legend(loc="lower left", ncol=3, fontsize=8)
fig.suptitle("Indices over the years (whole AOI)")
plt.tight_layout()
plt.show()

years = sorted(set(stats_all.index.year))
year_colors = dict(zip(years, plt.cm.viridis(np.linspace(0, 1, len(years)))))
fig, axes = plt.subplots(1, len(INDICES), figsize=(4.5 * len(INDICES), 3.8), squeeze=False)
for ax, name in zip(axes[0], INDICES):
    s = smoothed[name]
    for year, color in year_colors.items():
        y = s[s.index.year == year]
        ax.plot(y.index.dayofyear, y, color=color, lw=1.2, ls="--" if year in INCOMPLETE else "-", label=str(year))
    ax.set_xlim(*SEASON_WINDOW)
    ax.set_ylim(*clear_limits(name))
    ax.set_title(name)
    ax.set_xlabel("day of year")
    ax.grid(alpha=0.3)
axes[0, -1].legend(fontsize=7, ncol=2)
fig.suptitle(f"Seasonal course per year ({SMOOTH_WINDOW * 5}-day moving average, whole AOI; dashed: preliminary)")
plt.tight_layout()
plt.show()

## 3. Curves fitted per year and zone

Both date methods read their dates off a curve fitted to the clear observations of each index, zone and year:

| Curve | Fit | Source |
|---|---|---|
| `Whittaker` | weighted Whittaker smoother, daily, over the whole series (3b) | Eilers 2003; Atzberger & Eilers 2011 |
| `double logistic` | whole snow-free season, base fixed, weighted (3c) | Beck et al. 2006; TIMESAT / HR-VPP (Jönsson & Eklundh 2004) |

Curves are fitted for every index of `INDICES`. 3b and 3c each end with a figure of their curves for the whole AOI: one row per index, one column per year (`*` = preliminary), with the clear dates (× = weight 0: snow or outlier), the post-season level (dotted) and the method's start / mid / end (vertical lines). All dates, per index and combined per date type, go into one table, `season_dates` (3d). Overview of the methods: Zeng et al. 2020.

### 3a. Observations, weights, outliers and the post-season level

- **Weights** follow the HR-VPP quality classes: 1 for a good date (clear share ≥ `GOOD_VALID_FRACTION`), 0.5 for a usable one, 0 for a date with snow on more than `MAX_SNOW_FRACTION` of the zone.
- **Outliers** follow the HR-VPP rule: a date is dropped (weight 0) if its residual from a stiff smoothing curve exceeds `OUTLIER_FACTOR` × the global and × the local (91-day) median residual.
- **Post-season level ("base"):** the snow-free level after needle fall, the median of the clear dates in `POST_WINDOW` over all years. It is held fixed in the fits instead of being fitted per year, as Beck et al. (2006) do with the winter NDVI. Without a fixed base, a year with no data after the decline leaves the lower level unconstrained. Zones with fewer than `MIN_BASE_OBS` such dates use the whole AOI's level.
- **QA** of both methods (adapted from HR-VPP, ATBD Table 3): clear dates are counted on the plateau (30 days before 80 % of the amplitude is left), in the decline (80 % → 20 %) and after it (30 days past 20 %). high = more than 2 dates in all three parts, medium = in two, low = in one, filled = in none.

In [ ]:
# Both methods are fitted to every index. Per date type, a method's candidate is the median of its dates over the
# indices of SEASON_INDICES (index label "GRVI+RCC"); with a single index it is that index's date
METHODS = ["Whittaker", "double logistic"]
METRIC = {"start": "start", "peak": "mid", "end": "end"}   # date type -> share of the amplitude left (AMPLITUDE_FRACTIONS)
DATE_TYPES = list(SEASON_INDICES)
CURVE_INDICES = INDICES
ROLE = {}
for target, names in SEASON_INDICES.items():
    have = [n for n in names if n in INDICES]
    if target not in METRIC or not have:
        raise ValueError(f"SEASON_INDICES[{target!r}]: needs a date type of {list(METRIC)} and an index of {INDICES}")
    if len(have) < len(names):
        print(f"{target}: {sorted(set(names) - set(have))} not in the cache, using {have}")
    ROLE[target] = "+".join(have)
CANDIDATES = [(target, method, ROLE[target], METRIC[target]) for target in DATE_TYPES for method in METHODS]
print("Indices per date type:", ROLE)


def obs_frame(name, zone):
    # Clear observations of one index and zone: y (zone median), w (weight), year, doy. Weights as the HR-VPP
    # quality classes (ATBD; Tian et al. 2021): 1 = good, 0.5 = usable, 0 = snow (kept for the plots, not fitted)
    f = clean_frame(stats_all, name.lower(), zone)
    obs = pd.DataFrame({"y": f[f"{name.lower()}_median"],
                        "w": np.where(f.valid_fraction >= GOOD_VALID_FRACTION, 1.0, 0.5)}, index=f.index)
    obs.loc[f.scl_snow_fraction > MAX_SNOW_FRACTION, "w"] = 0.0
    obs = obs.dropna(subset=["y"])
    return obs.assign(year=obs.index.year, doy=obs.index.dayofyear, outlier=False)


def base_levels(name):
    # Post-season level per zone, held fixed in all fits (Beck et al. 2006): median of the weighted dates in
    # POST_WINDOW over all years. Fallbacks: the whole AOI's level; then, as HR-VPP sets its base level, the 5th
    # percentile of all clear dates (95th for rising indices, whose post-season level is the high one)
    rows = {}
    for zone in ZONES:
        o = obs_by_zone[name][zone]
        post = o[o.doy.between(*POST_WINDOW) & (o.w > 0)]
        rows[zone] = {"level": post.y.median(), "n_dates": len(post), "source": "zone"}
    for zone, r in rows.items():
        if r["n_dates"] < MIN_BASE_OBS and rows["all"]["n_dates"] >= MIN_BASE_OBS:
            r.update(level=rows["all"]["level"], source="whole AOI")
        elif r["n_dates"] < MIN_BASE_OBS:
            q = 0.05 if EXPECT_DECREASE[name] else 0.95
            r.update(level=obs_by_zone[name][zone].y.quantile(q), source=f"{q:.0%} percentile")
    return pd.DataFrame(rows).T.astype({"level": float, "n_dates": int})


# ---- Weighted Whittaker smoother (also used for the outlier test, so it is defined here)
def whittaker(days, y, w, lam, n):
    # Eilers (2003), for vegetation indices Atzberger & Eilers (2011): on a daily grid of n days, minimise
    # sum w (y - z)^2 + lam * sum (2nd difference of z)^2. Days without an observation have weight 0, so irregular
    # dates and gaps need no interpolation beforehand; across a gap the curve becomes nearly a straight line.
    W, Y = np.zeros(n), np.zeros(n)
    np.add.at(W, days, w)
    np.add.at(Y, days, w * y)
    D = sparse.diags([1.0, -2.0, 1.0], [0, 1, 2], shape=(n - 2, n))
    return spsolve((sparse.diags(W) + lam * (D.T @ D)).tocsc(), Y)


def smooth_series(obs, lam, base):
    # Daily Whittaker curve (pd.Series over whole calendar years) through the weighted dates of obs. Long
    # off-season gaps are filled as in HR-VPP: if more than 90 days pass without a good date between the autumn
    # and the spring equinox, control points at the base level (weight 0.5) are added every 15 days.
    o = obs[obs.w > 0]
    start = pd.Timestamp(obs.index.min().year, 1, 1)
    end = pd.Timestamp(obs.index.max().year, 12, 31)
    grid = pd.date_range(start, end)
    days, y, w = (o.index - start).days.values, o.y.values, o.w.values
    control = [d for a, b in zip(days[:-1], days[1:]) if b - a > 90 for d in range(a + 15, b - 7, 15)
               if not 80 < grid[d].dayofyear < 266]
    days = np.concatenate([days, control]).astype(int)
    y = np.concatenate([y, np.full(len(control), base)])
    w = np.concatenate([w, np.full(len(control), 0.5)])
    return pd.Series(whittaker(days, y, w, lam, len(grid)), index=grid)


def flag_outliers(obs, base):
    # HR-VPP outlier rule (ATBD 4.2.1): residuals from a stiff curve; outlier if the residual exceeds
    # OUTLIER_FACTOR x the global median residual and OUTLIER_FACTOR x the median of the surrounding 91 days
    o = obs[obs.w > 0]
    curve = smooth_series(obs, max(WHITTAKER_LAMBDAS), base)
    res = (o.y - curve.reindex(o.index).values).abs()
    local = res.rolling("91D", center=True, min_periods=1).median()
    out = res.index[(res > OUTLIER_FACTOR * res.median()) & (res > OUTLIER_FACTOR * local)]
    obs = obs.copy()
    obs.loc[out, ["w", "outlier"]] = [0.0, True]
    return obs


# ---- Shared by 3b and 3c: QA of the dates and the curve figure
QA_LEVELS = ["filled", "low", "medium", "high"]


def double_logistic(t, m, S, mS, A, mA, b):
    # Beck et al. (2006): b = off-season level (fixed), m = summer level, S / A = spring / autumn inflection day,
    # mS / mA = their rates (1/day)
    return b + (m - b) * (1 / (1 + np.exp(-mS * (t - S))) + 1 / (1 + np.exp(mA * (t - A))) - 1)


def fraction_day(t0, s, f):
    # Day on a logistic when a share f of the amplitude is left
    return t0 + s * np.log((1 - f) / f)


def fit_double_logistic(doy, y, w, b):
    summer = y[(doy >= SUMMER_WINDOW[0]) & (doy <= SUMMER_WINDOW[1])]
    p0 = [np.clip(np.median(summer if len(summer) else y), -1.9, 1.9), 160, 0.1, 280, 0.1]
    bounds = ([-2, SEASON_WINDOW[0], 1 / 40, SUMMER_WINDOW[0], 1 / 40], [2, SUMMER_WINDOW[0], 1, SEASON_WINDOW[1], 1])
    popt, _ = curve_fit(lambda t, m, S, mS, A, mA: double_logistic(t, m, S, mS, A, mA, b), doy, y, p0=p0,
                        sigma=1 / np.sqrt(w), bounds=bounds, maxfev=20000)
    return popt


def hrvpp_qa(doy, t80, t20):
    # Clear dates on the plateau, in the decline and after it (see above): high / medium / low / filled
    parts = [(doy >= t80 - 30) & (doy < t80), (doy >= t80) & (doy <= t20), (doy > t20) & (doy <= t20 + 30)]
    return QA_LEVELS[sum(int(p.sum()) > 2 for p in parts)]


CURVE_DOY = np.arange(SEASON_WINDOW[0], SEASON_WINDOW[1] + 1)
DATE_STYLES = {"start": "--", "mid": "-", "end": "-."}


def plot_curves(method, curve, rows):
    # Figure of 3b / 3c: curve(name, year) -> the whole-AOI curve over DOY (or None), rows -> the method's dates
    dates = {(r["index"], r["year"], r["metric"]): r["doy"] for r in rows if r["zone"] == "all"}
    fig, axes = plt.subplots(len(CURVE_INDICES), len(years), figsize=(2.3 * len(years), 2.3 * len(CURVE_INDICES) + 0.8),
                             sharex=True, sharey="row", squeeze=False)
    for i, name in enumerate(CURVE_INDICES):
        o_all = obs_by_zone[name]["all"]
        for j, year in enumerate(years):
            ax = axes[i, j]
            o = o_all[o_all.year == year]
            ax.scatter(o[o.w > 0].doy, o[o.w > 0].y, s=6, color=INDEX_COLORS[name])
            ax.scatter(o[o.w == 0].doy, o[o.w == 0].y, s=12, marker="x", color=INDEX_COLORS[name])
            z = curve(name, year)
            if z is not None:
                ax.plot(z.index, z.values, color=METHOD_COLORS[method], lw=1.2)
            for metric, ls in DATE_STYLES.items():
                if (name, year, metric) in dates:
                    ax.axvline(dates[(name, year, metric)], color="black", ls=ls, lw=0.8)
            ax.axhline(base[name].level["all"], color="grey", ls=":", lw=0.8)
            ax.grid(alpha=0.3)
            if i == 0:
                ax.set_title(f"{year}*" if year in INCOMPLETE else str(year), fontsize=9)
            if i == len(CURVE_INDICES) - 1:
                ax.set_xlabel("DOY")
        axes[i, 0].set_ylabel(name)
        axes[i, 0].set_ylim(*clear_limits(name))
    axes[0, 0].set_xlim(*SEASON_WINDOW)
    handles = [plt.Line2D([], [], color=METHOD_COLORS[method], lw=1.5, label="curve")]
    handles += [plt.Line2D([], [], color="black", ls=ls, lw=0.8, label=metric) for metric, ls in DATE_STYLES.items()]
    fig.legend(handles=handles, loc="lower center", ncol=len(handles), fontsize=8)
    fig.suptitle(f"{method} curves per year (whole AOI)")
    plt.tight_layout(rect=(0, 0.04, 1, 1))
    plt.show()


obs_by_zone = {name: {zone: obs_frame(name, zone) for zone in ZONES} for name in INDICES}
base = {name: base_levels(name) for name in INDICES}
for name in INDICES:
    for zone in ZONES:
        obs_by_zone[name][zone] = flag_outliers(obs_by_zone[name][zone], base[name].level[zone])

display(pd.concat({name: b for name, b in base.items()}, axis=1).round(3))
print("Dates per index and zone: weight 1 / 0.5 / 0 (snow) / outliers")
display(pd.DataFrame({name: {zone: f"{(o.w == 1).sum()} / {(o.w == 0.5).sum()} / {((o.w == 0) & ~o.outlier).sum()} / "
                                   f"{o.outlier.sum()}" for zone, o in by_zone.items()}
                      for name, by_zone in obs_by_zone.items()}))

### 3b. Whittaker smoother and its dates

The weighted Whittaker smoother (Eilers 2003) fits a smooth daily curve to the irregular observations; Atzberger & Eilers (2011) introduced it for vegetation-index time series. It needs no gap interpolation beforehand, and one parameter, λ, sets the smoothness. λ is chosen per index from `WHITTAKER_LAMBDAS` by 10-fold cross-validation on the whole AOI (weighted RMSE of left-out dates), then used for every zone. HR-VPP assumes clear Sentinel-2 dates are unbiased and applies no upper-envelope correction; neither does this notebook.

**Dates.** Per year, the summer level `hi` is the curve's median in `SUMMER_WINDOW`; the base is the post-season level (3a). With `r` = share of the amplitude left:
- mid = first day after the summer window with `r ≤ 0.5`
- start = last day before mid with `r ≥ 0.9` (MCD12Q2: "last date above 90 %")
- end = first day after mid with `r ≤ 0.15`

A date that is not reached within the year stays empty. The QA (3a) uses the curve's 80 % and 20 % days.

In [ ]:
def choose_lambda(obs, base_level, k=10, seed=0):
    # Weighted RMSE of left-out good dates for every lam (k-fold cross-validation)
    good = obs[obs.w > 0]
    fold = np.random.default_rng(seed).integers(0, k, len(good))
    rmse = {}
    for lam in WHITTAKER_LAMBDAS:
        sq = []
        for i in range(k):
            test = good[fold == i]
            curve = smooth_series(obs.drop(index=test.index), lam, base_level)
            sq.append(test.w * (test.y - curve.reindex(test.index).values) ** 2)
        rmse[lam] = np.sqrt(pd.concat(sq).sum() / good.w.sum())
    return pd.Series(rmse)


cv_lambda = pd.DataFrame({name: choose_lambda(obs_by_zone[name]["all"], base[name].level["all"]) for name in CURVE_INDICES})
LAMBDA = cv_lambda.idxmin()
print("Cross-validated RMSE per lambda (whole AOI); chosen:", LAMBDA.to_dict())
display(cv_lambda.rename_axis("lambda").round(4))
wcurves = {name: {zone: smooth_series(obs_by_zone[name][zone], LAMBDA[name], base[name].level[zone]) for zone in ZONES}
           for name in CURVE_INDICES}

In [ ]:
def whittaker_dates(curve, hi, lo):
    # Start / mid / end on one year of a daily curve (index = DOY); also the 80 % / 20 % days for the QA
    r = (curve - lo) / (hi - lo)
    below_mid = r[(r.index > SUMMER_WINDOW[1]) & (r <= AMPLITUDE_FRACTIONS["mid"])]
    if below_mid.empty:
        return None
    mid = below_mid.index[0]

    def last_above(f):
        above = r[(r.index < mid) & (r >= f)]
        return above.index[-1] if not above.empty else np.nan

    def first_below(f):
        below = r[(r.index > mid) & (r <= f)]
        return below.index[0] if not below.empty else np.nan

    days = {"start": last_above(AMPLITUDE_FRACTIONS["start"]), "mid": mid, "end": first_below(AMPLITUDE_FRACTIONS["end"])}
    return days, last_above(0.8), first_below(0.2)


whittaker_rows = []
for name in CURVE_INDICES:
    for zone in ZONES:
        obs, lo = obs_by_zone[name][zone], base[name].level[zone]
        curve_all = wcurves[name][zone]
        for year in sorted(obs.year.unique()):
            curve = curve_all[curve_all.index.year == year]
            curve.index = curve.index.dayofyear
            doy = obs[(obs.year == year) & (obs.w > 0)].doy.values
            hi = curve.loc[SUMMER_WINDOW[0]:SUMMER_WINDOW[1]].median()
            found = whittaker_dates(curve, hi, lo) if (hi > lo) == EXPECT_DECREASE[name] else None
            if found:
                days, t80, t20 = found
                qa = hrvpp_qa(doy, t80, t20) if np.isfinite([t80, t20]).all() else "filled"
                whittaker_rows += [{"index": name, "zone": zone, "year": year, "method": "Whittaker", "metric": m,
                                    "doy": d, "qa": qa, "at_bound": False, "n_obs": len(doy)} for m, d in days.items()]
print(f"{len(whittaker_rows)} Whittaker dates")


def whittaker_curve(name, year):
    z = wcurves[name]["all"]
    z = z[z.index.year == year]
    return pd.Series(z.values, index=z.index.dayofyear)


plot_curves("Whittaker", whittaker_curve, whittaker_rows)

### 3c. Double logistic and its dates

- **Fit:** the function of Beck et al. (2006) over the whole snow-free season (`SEASON_WINDOW`), `y = b + (m − b)·(1/(1 + exp(−mS(t − S))) + 1/(1 + exp(mA(t − A))) − 1)`, with the base `b` fixed at the post-season level (3a) and the dates weighted. It is the function type TIMESAT fits for HR-VPP; only its autumn half (`A`, `mA`) gives dates here. A year needs at least `MIN_OBS_PER_YEAR` weighted dates between `AUTUMN_START_DOY` and the end of `POST_WINDOW`.
- **Dates:** a share `f` of the amplitude is left at `t = A + ln((1 − f) / f) / mA`; the 50 % point is `A`.
- **QA** (3a) on the curve's 80 % and 20 % days.
- **Bounds:** a fit with `A` or `mA` on its bound gives dates that are not trusted (3d).

In [ ]:
def on_bound(value, low, high):
    return bool(np.isclose(value, low, atol=1e-3) or np.isclose(value, high, atol=1e-3))


dl_rows, dl_fits = [], {}
with warnings.catch_warnings():
    warnings.simplefilter("ignore", OptimizeWarning)
    for name in CURVE_INDICES:
        for zone in ZONES:
            obs, lo = obs_by_zone[name][zone], base[name].level[zone]
            for year, o in obs[obs.w > 0].groupby("year"):
                autumn = o[o.doy.between(AUTUMN_START_DOY, POST_WINDOW[1])]
                if len(autumn) < MIN_OBS_PER_YEAR:
                    continue
                season = o[o.doy.between(*SEASON_WINDOW)]
                try:
                    m, S, mS, A, mA = fit_double_logistic(season.doy.values.astype(float), season.y.values,
                                                          season.w.values, lo)
                except RuntimeError:
                    continue
                dl_fits[(name, zone, year)] = (m, S, mS, A, mA, lo)
                qa = hrvpp_qa(autumn.doy.values, fraction_day(A, 1 / mA, 0.8), fraction_day(A, 1 / mA, 0.2))
                bound = on_bound(A, SUMMER_WINDOW[0], SEASON_WINDOW[1]) or on_bound(mA, 1 / 40, 1)
                dl_rows += [{"index": name, "zone": zone, "year": year, "method": "double logistic", "metric": metric,
                             "doy": fraction_day(A, 1 / mA, f), "qa": qa, "at_bound": bound, "n_obs": len(season)}
                            for metric, f in AMPLITUDE_FRACTIONS.items()]
print(f"{len(dl_fits)} double-logistic fits")


def dl_curve(name, year):
    p = dl_fits.get((name, "all", year))
    return None if p is None else pd.Series(double_logistic(CURVE_DOY, *p), index=CURVE_DOY)


plot_curves("double logistic", dl_curve, dl_rows)

### 3d. All dates in one table

`season_dates` holds, per zone and year (long format):
- the dates of both methods for **every index** and metric (start, mid, end);
- per date type, the **candidate** of each method: the median over the indices of `SEASON_INDICES`, labelled e.g. `GRVI+RCC` (only for date types with more than one index; otherwise the single-index row is the candidate).

| Column | Meaning |
|---|---|
| `method`, `index`, `metric` | the date; metric = start, mid or end |
| `doy`, `date` | the estimated date |
| `n_obs` | clear dates of the year used for the curve (combined dates: the smallest of their indices) |
| `n_indices`, `index_spread` | combined dates: number of indices with a date and the days between the earliest and the latest |
| `trusted` | a clear date within `GAP_DAYS` on both sides (not inside a data gap), QA (3a) not low / filled and, for the double logistic, no parameter on its bound. A combined date is trusted if more than half of its indices give a trusted date |
| `preliminary` | year in `INCOMPLETE` |

In [ ]:
KEYS = ["method", "index", "metric"]
season_dates = pd.DataFrame(dl_rows + whittaker_rows).dropna(subset=["doy"]).reset_index(drop=True)


def bracket_days(group):
    # Days between the last clear date before and the first after each estimate (inf if none on a side)
    name, zone, year = group.name
    o = obs_by_zone[name][zone]
    doy = np.concatenate([[-np.inf], np.sort(o[(o.year == year) & (o.w > 0)].doy.values.astype(float)), [np.inf]])
    i = np.searchsorted(doy, group.doy.values, side="right")
    return pd.Series(doy[i] - doy[i - 1], index=group.index)


gap = season_dates.groupby(["index", "zone", "year"], group_keys=False)[["doy"]].apply(bracket_days)
season_dates["trusted"] = (gap <= GAP_DAYS) & ~season_dates.qa.isin(["low", "filled"]) & ~season_dates.at_bound.eq(True)
season_dates = season_dates.drop(columns=["qa", "at_bound"]).assign(n_indices=1)


def combined_dates(target):
    # Candidate of each method for one date type: median over its indices; trusted if more than half of them give a
    # trusted date
    names = ROLE[target].split("+")
    parts = season_dates[season_dates["index"].isin(names) & (season_dates.metric == METRIC[target])]
    out = parts.groupby(["method", "zone", "year"]).agg(
        doy=("doy", "median"), n_obs=("n_obs", "min"), n_indices=("index", "size"),
        index_spread=("doy", lambda x: x.max() - x.min()), trusted=("trusted", "sum"))
    out["trusted"] = out.trusted > len(names) / 2
    return out.reset_index().assign(index=ROLE[target], metric=METRIC[target])


season_dates = pd.concat([season_dates] + [combined_dates(t) for t in DATE_TYPES if "+" in ROLE[t]], ignore_index=True)
season_dates["preliminary"] = season_dates.year.isin(INCOMPLETE)
season_dates["date"] = pd.to_datetime(season_dates.year.astype(str)) + pd.to_timedelta(season_dates.doy.round() - 1, unit="D")
season_dates = season_dates.sort_values(["metric", *KEYS[:2], "zone", "year"]).reset_index(drop=True)
print(f"{len(season_dates)} dates in season_dates")

## 4. Start of the yellowing season

**Start** = the needles begin to lose chlorophyll: greenness leaves its summer level. Indices: `SEASON_INDICES["start"]` (GRVI, RCC).

| Method | Start = |
|---|---|
| `Whittaker` | last day before the 50 % point with ≥ 90 % of the amplitude left (MCD12Q2 "senescence"; Gray et al. 2019) |
| `double logistic` | 90 % of the amplitude left |

There is no independent reference for the start; section 7 checks it only for its order (before the peak) and data support.

**How sections 4–6 show the dates:**
- **Table:** the whole-AOI day of year per year (`*` = preliminary) for the two candidates and the single indices they combine, and `trusted_share`, the share of the zone dates that are `trusted` (3d).
- **Figure:** the whole-AOI dates per year: the candidates as thick lines (hollow = not trusted), the single indices dotted. Where they exist, the MeteoSwiss station dates are drawn in black. The stations lie lower than the AOI, so compare the year-to-year ups and downs, not the level.

In [ ]:
VALIDATE_WITH = {"peak": "mlard94d", "end": "mlard95d"}      # MeteoSwiss phase per date type
AOI_ELEV = zone_table.elev_mean["all"]   # mean elevation of the AOI pixels, shown beside the station mean
complete_years = [y for y in years if y not in INCOMPLETE]

# ---- validation data from 02_download_validation (skipped if its file is missing)
pheno = None
if (VALIDATION_DIR / "meteoswiss_larch.csv").exists():
    p = pd.read_csv(VALIDATION_DIR / "meteoswiss_larch.csv")
    p = p[p.year.isin(complete_years)]
    p["anomaly"] = p.doy - p.groupby(["station", "param"]).doy.transform("mean")   # each station vs its own mean
    pheno = p.groupby(["param", "year"]).agg(anomaly=("anomaly", "mean"), doy=("doy", "mean"),
                                            n_stations=("station", "nunique"), height=("height_masl", "mean"))
else:
    print("No MeteoSwiss observations (run 02_download_validation)")


def reference_dates(target):
    # [(label, line style, DOY per year)] of the validation data of one date type (whole AOI)
    refs = []
    if pheno is not None and target in VALIDATE_WITH:
        st = pheno.xs(VALIDATE_WITH[target], level="param")
        refs.append((f"MeteoSwiss {PHENO_PARAMS[VALIDATE_WITH[target]]} (stations, mean {st.height.mean():.0f} m)",
                     "ks--", st.doy))
    return refs


def show_candidates(target):
    # Table and figure of the two candidates of one date type and the single-index dates they combine (whole AOI)
    shown = [ROLE[target], *ROLE[target].split("+")]
    d = season_dates[(season_dates.metric == METRIC[target]) & season_dates["index"].isin(shown)]
    aoi = d[d.zone == "all"]
    table = aoi.pivot_table(index=KEYS, columns="year", values="doy").round(0)
    table.columns = [f"{y}*" if y in INCOMPLETE else str(y) for y in table.columns]
    zones = d[(d.zone != "all") & ~d.preliminary]
    table["trusted_share"] = zones.groupby(KEYS).trusted.mean().round(2)
    display(table.loc[sorted(table.index, key=lambda k: (k[1] != ROLE[target], k))])

    fig, ax = plt.subplots(figsize=(9, 4))
    for (method, index, metric), g in aoi.groupby(KEYS):
        g = g.sort_values("year")
        color, marker = METHOD_COLORS[method], INDEX_MARKERS.get(index, "D")
        if index != ROLE[target]:   # a single index of a combined candidate
            ax.plot(g.year, g.doy, ls=":", marker=marker, color=color, lw=1, ms=4, alpha=0.6,
                    label=f"{method} / {index}")
            continue
        ax.plot(g.year, g.doy, color=color, lw=2, label=f"{method} / {index} / {metric}")
        for trusted, mfc in [(True, color), (False, "white")]:
            t = g[g.trusted == trusted]
            ax.plot(t.year, t.doy, ls="", marker=marker, color=color, mfc=mfc, ms=7)
    for label, style, s in reference_dates(target):
        ax.plot(s.index, s.values, style, lw=1.5, ms=5, label=label)
    ax.plot([], [], ls="", marker="D", color="grey", mfc="white", label="not trusted")
    for year in INCOMPLETE:
        ax.axvspan(year - 0.5, year + 0.5, color="grey", alpha=0.1, lw=0)
    ax.set_title(f"{target.capitalize()}: whole-AOI dates per year (AOI mean {AOI_ELEV:.0f} m; shaded: preliminary)")
    ax.set_xlabel("year")
    ax.set_ylabel("day of year")
    ax.xaxis.get_major_locator().set_params(integer=True)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5))
    plt.tight_layout()
    plt.show()


show_candidates("start")

## 5. Peak of the yellowing

**Peak** = the canopy is most yellow: chlorophyll gone, needles still on. Both methods take the 50 % point of the autumn change, the MCD12Q2 "greendown midpoint" (Gray et al. 2019). Indices: `SEASON_INDICES["peak"]` (GRVI, RCC).

| Method | Peak = |
|---|---|
| `Whittaker` | first day after the summer window with ≤ 50 % of the amplitude left |
| `double logistic` | the autumn inflection day `A` |

Reference: MeteoSwiss needle colouring 50 % (`mlard94d`).

In [ ]:
show_candidates("peak")

## 6. End of the season

**End** = the needles have fallen: the index reaches its post-season level. Indices: `SEASON_INDICES["end"]` (NDVI, GRVI, RCC).

| Method | End = |
|---|---|
| `Whittaker` | first day after the 50 % point with ≤ 15 % of the amplitude left (HR-VPP end of season; Tian et al. 2021) |
| `double logistic` | 15 % of the amplitude left |

Reference: MeteoSwiss needle drop 50 % (`mlard95d`). There are few clear dates after mid-October, so many end dates lie in a data gap (low `trusted_share`).

In [ ]:
show_candidates("end")

## 7. Comparison of the candidates and the single indices

One table per date type: the two candidates (`candidate` = True) and, for comparison, both methods on every single index at the date type's metric. Complete years only.

| Column | Meaning |
|---|---|
| `coverage` | share of zone-years with a date |
| `trusted` | share of the zone dates that are `trusted` (3d) |
| `index_spread` | candidates combining several indices: median days between the earliest and the latest index (zone-years) |
| `order_ok` | share of zone-years in the order start < peak < end, judged against the median of the candidates of the neighbouring date types |
| `days_per_100m` | within-year elevation gradient across zones; negative = earlier higher up, as expected (elevation controls larch phenology; Migliavacca et al. 2008) |
| `aoi_sd_years` | year-to-year standard deviation of the whole-AOI date |
| `ms_r`, `ms_rmse`, `ms_n` | whole-AOI anomalies vs MeteoSwiss station anomalies: needle colouring 50 % for the peak, needle drop 50 % for the end. Correlation, RMSE in days, number of years |
| `ms_offset` | mean AOI date − mean station date. The stations lie lower (1050–1930 m), so this is not an error |

**Figure:** the whole-AOI mean date ± sd over the years per row of the table; diamonds = candidates; vertical lines = mean date of the reference data. With about 9 years, a correlation needs |r| > 0.67 to be significant at 5 %.

In [ ]:
d = season_dates[~season_dates.preliminary].dropna(subset=["doy"])
complete_years = [y for y in sorted(set(stats_all.index.year)) if y not in INCOMPLETE]
n_zone_years = (len(ZONES) - 1) * len(complete_years)
# ---- dates per (date type, method, index, metric): the candidates and every single index at the date type's metric
series = {(target, *key): g.set_index(["zone", "year"]).doy
          for target in DATE_TYPES for key, g in d[d.metric == METRIC[target]].groupby(KEYS)}
series = {k: series[k] for k in sorted(series, key=lambda k: (DATE_TYPES.index(k[0]), k not in CANDIDATES, k[2], k[1]))}
candidates = {k: s for k, s in series.items() if k in CANDIDATES}
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)   # zone-years without any candidate of a date type
    consensus = pd.DataFrame({t: pd.concat([s for k, s in candidates.items() if k[0] == t], axis=1).median(axis=1)
                              for t in DATE_TYPES})


def order_share(target, s):
    c = consensus.reindex(s.index)
    ok = {"start": s < c["peak"], "peak": (c["start"] < s) & (s < c["end"]), "end": s > c["peak"]}[target]
    known = {"start": c["peak"].notna(), "peak": c["start"].notna() & c["end"].notna(), "end": c["peak"].notna()}[target]
    return ok[known].mean() if known.any() else np.nan


def elevation_slope(s):
    # Days per 100 m from zone dates, within years (each year's mean removed from date and elevation)
    z = s.drop(index="all", level="zone", errors="ignore").reset_index()
    z["x"] = z.zone.map(zone_table.elev_mean) / 100
    z = z[z.groupby("year").x.transform("size") >= 2]
    x, y = z.x - z.groupby("year").x.transform("mean"), z.doy - z.groupby("year").doy.transform("mean")
    return (x @ y) / (x @ x) if len(z) > 3 and (x @ x) > 0 else np.nan


def meteoswiss_check(target, aoi):
    if pheno is None or target not in VALIDATE_WITH:
        return {}
    st = pheno.xs(VALIDATE_WITH[target], level="param")
    common = aoi.index.intersection(st.index)
    if len(common) < 4:
        return {"ms_n": len(common)}
    a = aoi[common] - aoi[common].mean()
    s = st.anomaly[common] - st.anomaly[common].mean()
    return {"ms_r": a.corr(s), "ms_rmse": np.sqrt(((a - s) ** 2).mean()), "ms_n": len(common),
            "ms_offset": (aoi[common] - st.doy[common]).mean()}


rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)   # medians / means of candidates without any value
    for (target, method, index, metric), s in series.items():
        g = d[(d.method == method) & (d["index"] == index) & (d.metric == metric)]
        gz = g[g.zone != "all"]
        aoi = s.xs("all", level="zone") if "all" in s.index.get_level_values("zone") else pd.Series(dtype=float)
        rows.append({"target": target, "method": method, "index": index, "metric": metric,
                     "candidate": (target, method, index, metric) in CANDIDATES,
                     "coverage": len(gz) / n_zone_years, "trusted": gz.trusted.mean(),
                     "index_spread": gz.index_spread.median(),
                     "order_ok": order_share(target, s.drop(index="all", level="zone", errors="ignore")),
                     "days_per_100m": elevation_slope(s), "aoi_mean_doy": aoi.mean(), "aoi_sd_years": aoi.std(),
                     **meteoswiss_check(target, aoi)})
comparison = pd.DataFrame(rows).set_index(["target", *KEYS])
display(comparison.round(2))

# ---- figure: mean date +- sd over the years per row (whole AOI)
n_rows = comparison.groupby(level="target").size().max()
fig, axes = plt.subplots(1, len(DATE_TYPES), figsize=(5.5 * len(DATE_TYPES), 0.3 * n_rows + 1.2), squeeze=False)
for ax, target in zip(axes[0], DATE_TYPES):
    t = comparison.xs(target, level="target")
    for k, ((method, index, metric), r) in enumerate(t.iterrows()):
        ax.errorbar(r.aoi_mean_doy, k, xerr=r.aoi_sd_years, fmt="D" if r.candidate else INDEX_MARKERS.get(index, "o"),
                    color=METHOD_COLORS[method], ms=7 if r.candidate else 5, alpha=1 if r.candidate else 0.6, capsize=3)
    for (label, style, s), ls in zip(reference_dates(target), ["--", ":"]):
        ax.axvline(s.mean(), color="black", ls=ls, lw=1.2, label=label)
    ax.set_yticks(range(len(t)), [" / ".join(key) for key in t.index], fontsize=8)
    ax.set_ylim(len(t) - 0.4, -0.6)
    ax.set_title(f"{target}: mean ± sd over years (whole AOI, mean {AOI_ELEV:.0f} m)")
    ax.set_xlabel("day of year")
    ax.grid(alpha=0.3)
    if ax.get_legend_handles_labels()[0]:
        ax.legend(fontsize=7, loc="lower left")
plt.tight_layout()
plt.show()


### 7b. Chosen dates

`SEASON_CHOICE` in `config.py` sets the date used in sections 8 and 9, per date type:
- `"average"`: the mean of both candidates (one per method). Each enters as its anomaly from its own mean over all zone-years, plus the mean level of both, so a candidate missing in one year does not shift that year's level.
- one row of the table of section 7, e.g. `"Whittaker / GRVI+RCC / mid"` or a single index such as `"Whittaker / GRVI / mid"`.

`selected` holds the chosen start / peak / end per zone and year (complete years). The figure shows the season of the whole AOI per year with the reference data.

In [ ]:
def chosen_keys(target):
    choice = SEASON_CHOICE[target]
    if choice == "average":
        keys = [k[1:] for k in candidates if k[0] == target]
        if not keys:
            raise ValueError(f"No {target} candidate has dates; check SEASON_INDICES[{target!r}]")
        return keys
    key = tuple(part.strip() for part in choice.split("/"))
    if (target, *key) not in series:
        options = [" / ".join(k[1:]) for k in series if k[0] == target]
        raise ValueError(f"SEASON_CHOICE[{target!r}] = {choice!r} is not a {target} date with values. Options: {options}")
    return [key]


def combine(target, keys):
    # Mean of the chosen dates per zone-year: each one's anomaly from its own mean over all zone-years, averaged over
    # the dates present, plus the mean level of all of them
    s = pd.concat([series[(target, *k)] for k in keys], axis=1, ignore_index=True)
    return s.sub(s.mean()).mean(axis=1) + s.mean().mean()


def doy_to_date(year, doy):
    return date(int(year), 1, 1) + timedelta(days=int(round(doy)) - 1) if pd.notna(doy) else None


chosen = {target: chosen_keys(target) for target in DATE_TYPES}
choice_label = {t: f"average of {len(k)}" if len(k) > 1 else " / ".join(k[0]) for t, k in chosen.items()}
for target, keys in chosen.items():
    print(f"{target}: {choice_label[target]}: {'; '.join(' / '.join(k) for k in keys)}")
selected = pd.DataFrame({target: combine(target, keys) for target, keys in chosen.items()}).sort_index()

z = selected.drop(index="all", level="zone").dropna()
print(f"start < peak < end in {((z.start < z.peak) & (z.peak < z.end)).mean():.0%} of {len(z)} zone-years")
aoi_sel = selected.xs("all", level="zone")
display(aoi_sel.round(0).join(pd.DataFrame({f"{t}_date": [doy_to_date(y, v) for y, v in aoi_sel[t].items()]
                                            for t in DATE_TYPES}, index=aoi_sel.index)))

MONTH_TICKS = {date(2023, m, 1).timetuple().tm_yday: f"1 {date(2023, m, 1):%b}" for m in range(8, 13)}
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.vlines(aoi_sel.index, aoi_sel.start, aoi_sel.end, color="tab:olive", lw=10, alpha=0.35, label="start to end")
ax.plot(aoi_sel.index, aoi_sel.start, "_", color="tab:olive", ms=18, mew=2, label=f"start ({choice_label['start']})")
ax.plot(aoi_sel.index, aoi_sel.peak, "o", color="darkorange", ms=9, label=f"peak ({choice_label['peak']})")
ax.plot(aoi_sel.index, aoi_sel.end, "_", color="saddlebrown", ms=18, mew=2, label=f"end ({choice_label['end']})")
for target, marker in [("peak", "s"), ("end", "D")]:
    for label, _, s in reference_dates(target):
        ax.plot(s.index + 0.2, s.values, marker, color="black", mfc="white", ms=6, ls="", label=label)
ax.set_yticks(list(MONTH_TICKS), [f"{v} ({k})" for k, v in MONTH_TICKS.items()])
ax.set_xlabel("year")
ax.xaxis.get_major_locator().set_params(integer=True)
ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5))
ax.set_title(f"Chosen season dates per year (whole AOI, mean {AOI_ELEV:.0f} m; tick labels: date (DOY))")
plt.tight_layout()
plt.show()

## 8. Trend and short-term projection (no climate drivers)

For each chosen date (7b) of the whole AOI and the complete years:
- **OLS trend** of the day of year on the year, with its 95 % confidence interval, extrapolated `FORECAST_YEARS` ahead with a 95 % prediction interval (the range a single future year is expected to fall in).
- **Mean model** (no trend) alongside. If the trend is not significant (`trend_significant` = False), the mean ± interval is the more honest expectation.

With about 9 years, one early or late year at either end moves the slope a lot. This is the statistical baseline that the climate models in `05` are compared against.

In [ ]:
proj_rows = []
level = f"{1 - ALPHA:.0%}"
fig, axes = plt.subplots(1, len(DATE_TYPES), figsize=(5.5 * len(DATE_TYPES), 4.5), squeeze=False)
for ax, target in zip(axes[0], DATE_TYPES):
    d = aoi_sel[target].dropna()
    n = len(d)
    if n < 4:
        ax.set_title(f"{target}: only {n} years, skipped")
        continue
    years_fit = d.index.values.astype(float)
    y = d.values.astype(float)
    x0 = years_fit.mean()   # centred years for numerical stability
    ols = sm.OLS(y, sm.add_constant(years_fit - x0)).fit()
    mean_model = sm.OLS(y, np.ones((n, 1))).fit()
    future = np.arange(years_fit.max() + 1, years_fit.max() + 1 + FORECAST_YEARS)
    grid = np.arange(years_fit.min(), future.max() + 1)
    pred = ols.get_prediction(sm.add_constant(grid - x0, has_constant="add")).summary_frame(alpha=ALPHA)
    pred_mean = mean_model.get_prediction(np.ones((len(grid), 1))).summary_frame(alpha=ALPHA)
    slope_lo, slope_hi = ols.conf_int(ALPHA)[1]
    significant = bool(ols.pvalues[1] < ALPHA)
    print(f"{target} ({choice_label[target]}), {n} years: OLS slope {ols.params[1]:+.2f} d/yr "
          f"({level} CI {slope_lo:+.2f} to {slope_hi:+.2f}), p = {ols.pvalues[1]:.3f}, R2 = {ols.rsquared:.2f}")
    for i, yr in enumerate(grid):
        if yr in future:
            proj_rows.append({
                "date_type": target, "year": int(yr),
                "trend_doy": pred["mean"].iloc[i],
                "trend_pi_low": pred["obs_ci_lower"].iloc[i], "trend_pi_high": pred["obs_ci_upper"].iloc[i],
                "mean_doy": pred_mean["mean"].iloc[i],
                "mean_pi_low": pred_mean["obs_ci_lower"].iloc[i], "mean_pi_high": pred_mean["obs_ci_upper"].iloc[i],
                "trend_significant": significant,
            })

    color = "tab:blue"
    ax.fill_between(grid, pred["obs_ci_lower"], pred["obs_ci_upper"], color=color, alpha=0.15,
                    label=f"OLS {level} prediction interval")
    ax.fill_between(grid, pred["mean_ci_lower"], pred["mean_ci_upper"], color=color, alpha=0.3,
                    label=f"OLS {level} CI of the trend")
    ax.plot(grid, pred["mean"], color=color, lw=2, label="OLS trend")
    ax.plot(grid, pred_mean["mean"], "--", color="grey", lw=1, label="mean (no trend)")
    ax.fill_between(grid, pred_mean["obs_ci_lower"], pred_mean["obs_ci_upper"], facecolor="none",
                    edgecolor="grey", hatch="//", lw=0, alpha=0.5, label=f"mean {level} prediction interval")
    ax.plot(years_fit, y, "o", color="black", label="chosen date")
    ax.axvspan(future.min() - 0.5, future.max() + 0.5, color="grey", alpha=0.08)
    ax.set_title(f"{target} ({choice_label[target]}): {ols.params[1]:+.2f} d/yr, p = {ols.pvalues[1]:.2f}")
    ax.set_xlabel("year")
    ax.xaxis.get_major_locator().set_params(integer=True)
    ax.grid(alpha=0.3)
axes[0, 0].set_ylabel("day of year")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(handles), fontsize=8)
plt.suptitle("Trend and projection of the chosen dates (whole AOI; shaded columns = projected years)")
plt.tight_layout(rect=(0, 0.07, 1, 1))
plt.show()

season_projection = pd.DataFrame(proj_rows)
if not season_projection.empty:
    season_projection = season_projection.set_index(["date_type", "year"]).round(1)
    for c in ["trend_doy", "mean_doy"]:
        season_projection[c.replace("_doy", "_date")] = [doy_to_date(yr, v) for (_, yr), v in season_projection[c].items()]
    display(season_projection)

## 9. Phenology vs topography (elevation band × aspect)

Uses the chosen dates of 7b per zone and year.

- **Map:** elevation band and aspect class of the AOI pixels (`zones.tif` from `01`) on a hillshade. The hillshade is lit from the south, so north-facing slopes appear dark: blue (N) pixels should sit on the dark side of the valleys.
- **Table:** zone attributes, the mean number of clear autumn dates per year and index (`autumn_dates_*`, `AUTUMN_START_DOY`–`POST_WINDOW[1]`) and the mean share of SCL "dark" pixels in autumn.
- **Plot:** day of year against elevation band, one line per aspect (mean ± sd over years).
- **Model:** OLS `doy ~ elev_100m + C(aspect) + C(year)`.
  - The elevation coefficient is in days per 100 m.
  - The aspect coefficient is the north − south difference in days.
  - `C(year)` absorbs year-to-year weather.

**Caveats:**
- Zones share weather and clouds within a year, so the CIs are somewhat optimistic.
- North slopes lose more autumn dates to SCL class 2 (dark / topographic shadow); see `autumn_dates_*` and `autumn_dark_fraction`.

In [ ]:
zr = rioxarray.open_rasterio(CACHE_DIR / "zones.tif", masked=True)
elev, elev_band, aspect = (zr.sel(band=b).values for b in (1, 2, 3))
on = ~np.isnan(elev_band)
yy, xx = np.meshgrid(zr.y.values, zr.x.values, indexing="ij")
left, bottom, right, top = zr.rio.bounds()
shade = LightSource(azdeg=180, altdeg=30).hillshade(np.nan_to_num(elev, nan=np.nanmean(elev)),
                                                   dx=RESOLUTION, dy=RESOLUTION)
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
for ax in axes:
    ax.imshow(shade, cmap="gray", extent=(left, right, bottom, top), alpha=0.6)
bands = np.unique(elev_band[on])
for b, color in zip(bands, plt.get_cmap("viridis")(np.linspace(0, 1, len(bands)))):
    sel = on & (elev_band == b)
    axes[0].scatter(xx[sel], yy[sel], s=0.2, color=color, label=f"{int(b)}-{int(b + ELEV_STEP)} m")
for asp, code in ASPECT_CODES.items():
    sel = on & (aspect == code)
    axes[1].scatter(xx[sel], yy[sel], s=0.2, color=ASPECT_COLORS[asp], label=f"{asp} ({int(sel.sum())} px)")
for ax, title in zip(axes, ["Elevation band of the AOI pixels", "Aspect class (hillshade lit from the south)"]):
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.legend(markerscale=20, fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
zone_attrs = zone_table.drop(index="all")[["elev_band", "aspect", "elev_mean"]]
autumn = stats_all[stats_all.index.dayofyear >= AUTUMN_START_DOY]
autumn_dates = pd.DataFrame({
    f"autumn_dates_{name}": {zone: o[(o.w > 0) & o.doy.between(AUTUMN_START_DOY, POST_WINDOW[1]) & ~o.year.isin(INCOMPLETE)]
                             .groupby("year").size().mean() for zone, o in obs_by_zone[name].items()}
    for name in INDICES})
coverage = (zone_table[["elev_band", "aspect", "n_pixels", "elev_mean", "slope_mean", "northness_mean"]]
            .join(autumn_dates)
            .join(autumn.groupby("zone").scl_dark_fraction.mean().rename("autumn_dark_fraction")))
display(coverage.round(2))

zone_dates = selected.drop(index="all", level="zone").reset_index().join(zone_attrs, on="zone", how="inner")

fig, axes = plt.subplots(1, len(DATE_TYPES), figsize=(4.5 * len(DATE_TYPES), 4), sharex=True, squeeze=False)
for ax, target in zip(axes[0], DATE_TYPES):
    for asp, color in ASPECT_COLORS.items():
        d = zone_dates[zone_dates.aspect == asp].dropna(subset=[target])
        if d.empty:
            continue
        shift = ELEV_STEP / 2 + {"N": -0.08, "S": 0.08, "flat": 0}[asp] * ELEV_STEP   # band centre, dodged
        ax.scatter(d.elev_band + shift, d[target], s=8, color=color, alpha=0.3)
        g = d.groupby("elev_band")[target].agg(["mean", "std"])
        ax.errorbar(g.index + shift, g["mean"], yerr=g["std"].fillna(0), fmt="o-", color=color, capsize=3,
                    label=asp)
    ax.set_title(f"{target} ({choice_label[target]})")
    ax.set_xlabel("elevation band centre (m)")
    ax.grid(alpha=0.3)
axes[0, 0].set_ylabel("day of year")
axes[0, 0].legend(title="aspect")
plt.suptitle("Chosen dates vs elevation and aspect (points: zone-years, lines: mean ± sd over years)")
plt.tight_layout()
plt.show()

NS_TERM = "C(aspect, Treatment('S'))[T.N]"


def topo_model(data, col):
    d = data.dropna(subset=[col]).assign(elev_100m=lambda x: x.elev_mean / 100)
    if (not {"N", "S"} <= set(d.aspect) or d.elev_band.nunique() < 2
            or len(d) < d.year.nunique() + d.aspect.nunique() + 3):
        return None
    return smf.ols(f"{col} ~ elev_100m + C(aspect, Treatment('S')) + C(year)", data=d).fit()
rows = []
for target in DATE_TYPES:
    fit = topo_model(zone_dates, target)
    if fit is None:
        print(f"{target}: too few zone-years (needs N and S zones and >= 2 elevation bands), no model")
        continue
    ci = fit.conf_int(ALPHA)
    rows.append({
        "date_type": target, "method": choice_label[target], "n_zone_years": int(fit.nobs),
        "n_zones": zone_dates.dropna(subset=[target]).zone.nunique(),
        "days_per_100m": fit.params["elev_100m"],
        "elev_ci_low": ci.loc["elev_100m", 0], "elev_ci_high": ci.loc["elev_100m", 1],
        "elev_p": fit.pvalues["elev_100m"],
        "north_minus_south_days": fit.params[NS_TERM],
        "ns_ci_low": ci.loc[NS_TERM, 0], "ns_ci_high": ci.loc[NS_TERM, 1], "ns_p": fit.pvalues[NS_TERM],
        "r2": fit.rsquared,
    })
topo_effects = pd.DataFrame(rows).set_index("date_type").round(3) if rows else pd.DataFrame()
display(topo_effects)

## 10. Trend / seasonal / remainder decomposition (STL, whole AOI)

Robust STL on a regular 5-day grid splits each index into a long-term trend, a repeating yearly cycle and the remainder. It needs at least two full years.

In [ ]:
STEP_DAYS = 5
PERIOD = int(round(365.25 / STEP_DAYS))   # observations per year on the regular grid


def stl_decompose(series, name):
    s = series.resample(f"{STEP_DAYS}D").mean()
    s = s.loc[s.first_valid_index():s.last_valid_index()].interpolate(method="time")
    if len(s) < 2 * PERIOD:
        print(f"{name}: {len(s) * STEP_DAYS / 365.25:.1f} years of data; STL needs at least 2 full years")
        return None
    res = STL(s, period=PERIOD, robust=True).fit()
    return pd.DataFrame({"observed": s, "trend": res.trend, "seasonal": res.seasonal, "remainder": res.resid})


for name in INDICES:
    dec = stl_decompose(medians[name], name)
    if dec is None:
        continue
    color = INDEX_COLORS[name]
    fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
    axes[0].plot(dec.index, dec.observed, color="grey", lw=1)
    axes[0].set_ylabel(f"{name} observed")
    axes[1].plot(dec.index, dec.trend, color=color, lw=2)
    axes[1].set_ylabel("trend")
    axes[2].plot(dec.index, dec.seasonal, color=color, lw=1.5)
    axes[2].set_ylabel("seasonal")
    axes[3].scatter(dec.index, dec.remainder, s=6, color=color)
    axes[3].set_ylabel("remainder")
    for a in axes[2:]:
        a.axhline(0, color="black", lw=0.5)
    for a in axes:
        a.grid(alpha=0.3)
    axes[0].set_title(f"{name} STL decomposition (period = {PERIOD} x {STEP_DAYS} days)")
    plt.tight_layout()
    plt.show()
    slope = np.polyfit((dec.index - dec.index[0]).days / 365.25, dec.trend.values, 1)[0]
    print(f"{name} trend slope: {slope:+.4f} per year | seasonal amplitude: "
          f"{dec.seasonal.max() - dec.seasonal.min():.3f} | remainder std: {dec.remainder.std():.3f}")

## 11. Export

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_DIR / "ndvi_timeseries_all_dates.csv")
df_clean.to_csv(OUT_DIR / "ndvi_timeseries_clean.csv")
season_dates.to_csv(OUT_DIR / "season_dates.csv", index=False)
comparison.to_csv(OUT_DIR / "season_method_comparison.csv")
selected.to_csv(OUT_DIR / "season_selected.csv")
season_projection.to_csv(OUT_DIR / "season_projection.csv")
zone_table.to_csv(OUT_DIR / "zones.csv")
topo_effects.to_csv(OUT_DIR / "topography_effects.csv")
print(f"Results written to {OUT_DIR}")

## References

- Atzberger, C. & Eilers, P. H. C. (2011). A time series for monitoring vegetation activity and phenology at 10-daily time steps covering large parts of South America. *International Journal of Digital Earth* 4(5), 365–386.
- Beck, P. S. A., Atzberger, C., Høgda, K. A., Johansen, B. & Skidmore, A. K. (2006). Improved monitoring of vegetation dynamics at very high latitudes: a new method using MODIS NDVI. *Remote Sensing of Environment* 100(3), 321–334.
- Busetto, L., Colombo, R., Migliavacca, M., et al. (2010). Remote sensing of larch phenological cycle and analysis of relationships with climate in the Alpine region. *Global Change Biology* 16(9), 2504–2517.
- Copernicus Land Monitoring Service (2021). *HR-VPP Algorithm Theoretical Basis Document: Seasonal Trajectories and VPP parameters*, issue 1.1. https://land.copernicus.eu/en/technical-library
- Eilers, P. H. C. (2003). A perfect smoother. *Analytical Chemistry* 75(14), 3631–3636.
- Gray, J., Sulla-Menashe, D. & Friedl, M. A. (2019). *User Guide to Collection 6 MODIS Land Cover Dynamics (MCD12Q2) Product*. NASA LP DAAC.
- Jönsson, P. & Eklundh, L. (2004). TIMESAT – a program for analyzing time-series of satellite sensor data. *Computers & Geosciences* 30(8), 833–845.
- MeteoSwiss. Phenological observations, open data (`ch.meteoschweiz.ogd-phenology`). https://opendatadocs.meteoswiss.ch/a-data-groundbased/a9-phenological-observations
- Merzlyak, M. N., Gitelson, A. A., Chivkunova, O. B. & Rakitin, V. Y. (1999). Non-destructive optical detection of pigment changes during leaf senescence and fruit ripening. *Physiologia Plantarum* 106(1), 135–141.
- Migliavacca, M., Cremonese, E., Colombo, R., et al. (2008). European larch phenology in the Alps: can we grasp the role of ecological factors by combining field observations and inverse modelling? *International Journal of Biometeorology* 52(7), 587–605.
- Motohka, T., Nasahara, K. N., Oguma, H. & Tsuchida, S. (2010). Applicability of green-red vegetation index for remote sensing of vegetation phenology. *Remote Sensing* 2(10), 2369–2387.
- Richardson, A. D., Hufkens, K., Milliman, T., et al. (2018). Tracking vegetation phenology across diverse North American biomes using PhenoCam imagery. *Scientific Data* 5, 180028.
- Tian, F., Cai, Z., Jin, H., et al. (2021). Calibrating vegetation phenology from Sentinel-2 using eddy covariance, PhenoCam, and PEP725 networks across Europe. *Remote Sensing of Environment* 260, 112456.
- Zeng, L., Wardlow, B. D., Xiang, D., Hu, S. & Li, D. (2020). A review of vegetation phenological metrics extraction using time-series, multispectral satellite data. *Remote Sensing of Environment* 237, 111511.